# AeroPulse Bronze Airlines Ingestion

## Purpose

This notebook ingests airline reference data from the AeroPulse raw landing layer into the Bronze Delta Lake layer.

### Source

- Source system: ERP
- Entity: Airlines
- Source format: CSV

### Source location

`/Volumes/workspace/aeropulse_dev/raw_landing/erp/airlines/`

### Target

`workspace.aeropulse_dev.bronze_airlines`

### Bronze Responsibilities

- Read raw source data
- Preserve source-level business attributes
- Add ingestion metadata
- Maintain source lineage
- Write to a Delta Lake Bronze table

In [0]:
ENVIRONMENT = "dev"

CATALOG = "workspace"

SCHEMA = f"aeropulse_{ENVIRONMENT}"

SOURCE_SYSTEM = "erp"

SOURCE_ENTITY = "airlines"

RAW_LANDING_PATH = (
    f"/Volumes/{CATALOG}/{SCHEMA}/raw_landing"
)

SOURCE_PATH = (
    f"{RAW_LANDING_PATH}/{SOURCE_SYSTEM}/{SOURCE_ENTITY}"
)

BRONZE_TABLE = (
    f"{CATALOG}.{SCHEMA}.bronze_{SOURCE_ENTITY}"
)

print(f"Environment: {ENVIRONMENT}")
print(f"Source path: {SOURCE_PATH}")
print(f"Bronze table: {BRONZE_TABLE}")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_TABLE}
(
    airline_id STRING,
    airline_code STRING,
    airline_name STRING,
    country STRING,
    region STRING,
    founded_year STRING,
    fleet_size STRING,
    status STRING,
    source_updated_timestamp STRING,

    _ingestion_timestamp TIMESTAMP NOT NULL,
    _source_file_path STRING NOT NULL,
    _source_system STRING NOT NULL,
    _pipeline_run_id STRING NOT NULL
)
USING DELTA
""")

print(f"Bronze table ready: {BRONZE_TABLE}")

In [0]:
spark.sql(
    f"DESCRIBE TABLE {BRONZE_TABLE}"
).show(truncate=False)

In [0]:
spark.sql(
    f"SHOW TABLES IN {CATALOG}.{SCHEMA}"
).show(truncate=False) 

# Bronze Ingestion Pipeline

This section executes the first AeroPulse Bronze ingestion pipeline.

The pipeline performs the following steps:

1. Generate a unique pipeline execution ID.
2. Create a centralized audit record.
3. Read the raw CSV source from the landing Volume.
4. Add Bronze ingestion metadata.
5. Write the data to the Bronze Delta table.
6. Update the audit status to `SUCCESS` or `FAILED`.

In [0]:
import sys
sys.path.append('/Workspace/Users/hclearningtools08@gmail.com/aeropulse-databricks-lakehouse/src/audit')
from pipeline_audit import *

In [0]:
from uuid import uuid4

PIPELINE_RUN_ID = str(uuid4())

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")

In [0]:
AUDIT_TABLE = (
    f"{CATALOG}.{SCHEMA}.audit_pipeline_runs"
)

PIPELINE_NAME = "bronze_airlines_ingestion"

print(f"Audit table: {AUDIT_TABLE}")
print(f"Pipeline name: {PIPELINE_NAME}")

In [0]:
try:

    # Start centralized pipeline auditing.
    start_pipeline_run(
        spark=spark,
        audit_table=AUDIT_TABLE,
        pipeline_run_id=PIPELINE_RUN_ID,
        pipeline_name=PIPELINE_NAME,
        environment=ENVIRONMENT,
        layer="bronze",
        source_system=SOURCE_SYSTEM,
        target_table=BRONZE_TABLE,
    )

    print("Pipeline audit started.")

    # Read raw CSV source data from the Unity Catalog Volume.
    source_df = (
        spark.read
        .option("header", "true")
        .csv(SOURCE_PATH)
    )

    # Count source records for audit metrics.
    records_read = source_df.count()

    print(f"Records read: {records_read}")

    # Add Bronze technical metadata.
    from pyspark.sql import functions as F

    bronze_df = (
        source_df
        .withColumn(
            "_ingestion_timestamp",
            F.current_timestamp()
        )
        .withColumn(
            "_source_file_path",
            F.col("_metadata.file_path")
        )
        .withColumn(
            "_source_system",
            F.lit(SOURCE_SYSTEM)
        )
        .withColumn(
            "_pipeline_run_id",
            F.lit(PIPELINE_RUN_ID)
        )
    )

    # Write records to the Bronze Delta table.
    (
        bronze_df.write
        .format("delta")
        .mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    # Count successfully written records.
    records_inserted = bronze_df.count()

    print(f"Bronze ingestion completed successfully. Records inserted: {records_inserted}")


except Exception as error:

    error_message = str(error)

    print(f"Pipeline failed: {error_message}")

    # Re-raise the exception so the notebook execution fails visibly.
    raise

In [0]:
display(
    spark.table(BRONZE_TABLE)
)

In [0]:
bronze_count = spark.table(BRONZE_TABLE).count()

print(f"Bronze record count: {bronze_count}")

In [0]:
display(
    spark.sql(f"""
        SELECT
            airline_id,
            airline_name,
            _ingestion_timestamp,
            _source_file_path,
            _source_system,
            _pipeline_run_id
        FROM {BRONZE_TABLE}
        LIMIT 10
    """)
)

In [0]:
display(
    spark.sql(f"""
        SELECT
            pipeline_run_id,
            pipeline_name,
            environment,
            layer,
            pipeline_status,
            records_read,
            records_inserted,
            records_rejected,
            start_timestamp,
            end_timestamp,
            error_message
        FROM {AUDIT_TABLE}
        WHERE pipeline_run_id = '{PIPELINE_RUN_ID}'
    """)
)

In [0]:
INGESTION_FILE_REGISTRY_TABLE = (
    f"{CATALOG}.{SCHEMA}.ingestion_file_registry"
)

print(f"Ingestion registry: {INGESTION_FILE_REGISTRY_TABLE}")

# File-Level Delivery Discovery

This section discovers timestamped source deliveries from the raw landing layer.

Each delivery is checked against the ingestion registry to prevent duplicate processing during pipeline reruns.

In [0]:
all_items = dbutils.fs.ls(SOURCE_PATH)

delivery_paths = [
    item.path
    for item in all_items
    if item.isDir()
    and "airlines_" in item.name
]

print("Discovered source deliveries:")

for delivery_path in delivery_paths:
    print(delivery_path)

In [0]:
def is_delivery_processed(
    spark,
    registry_table,
    source_delivery_path,
):
    """
    Check whether a source delivery has already been
    successfully ingested.
    """

    escaped_path = source_delivery_path.replace("'", "''")

    result_df = spark.sql(f"""
        SELECT COUNT(*) AS processed_count
        FROM {registry_table}
        WHERE source_file_path = '{escaped_path}'
          AND ingestion_status = 'SUCCESS'
    """)

    processed_count = (
        result_df
        .collect()[0]["processed_count"]
    )

    return processed_count > 0

In [0]:
from uuid import uuid4

PIPELINE_RUN_ID = str(uuid4())

print(f"Pipeline Run ID: {PIPELINE_RUN_ID}")

In [0]:
from datetime import datetime
from pyspark.sql import functions as F


records_read_total = 0
records_inserted_total = 0
records_rejected_total = 0


try:

    # Start pipeline-level audit.
    start_pipeline_run(
        spark=spark,
        audit_table=AUDIT_TABLE,
        pipeline_run_id=PIPELINE_RUN_ID,
        pipeline_name=PIPELINE_NAME,
        environment=ENVIRONMENT,
        layer="bronze",
        source_system=SOURCE_SYSTEM,
        target_table=BRONZE_TABLE,
    )

    print("Pipeline audit started.")


    for delivery_path in delivery_paths:

        print(
            f"\nChecking source delivery: "
            f"{delivery_path}"
        )

        # Skip deliveries already processed successfully.
        if is_delivery_processed(
            spark=spark,
            registry_table=INGESTION_FILE_REGISTRY_TABLE,
            source_delivery_path=delivery_path,
        ):

            print(
                "Delivery already processed successfully. "
                "Skipping."
            )

            continue


        # Get delivery name for registry tracking.
        delivery_name = (
            delivery_path
            .rstrip("/")
            .split("/")[-1]
        )


        # Register delivery as RUNNING.
        spark.sql(f"""
            INSERT INTO {INGESTION_FILE_REGISTRY_TABLE}
            VALUES
            (
                '{delivery_path}',
                '{delivery_name}',
                '{SOURCE_SYSTEM}',
                '{SOURCE_ENTITY}',
                'CSV',

                '{PIPELINE_RUN_ID}',
                'RUNNING',

                NULL,
                NULL,

                current_timestamp(),
                NULL,

                NULL,

                current_timestamp(),
                current_timestamp()
            )
        """)

        print(
            f"Delivery registered as RUNNING: "
            f"{delivery_name}"
        )


        try:

            # Read the individual source delivery.
            source_df = (
                spark.read
                .option("header", "true")
                .csv(delivery_path)
            )

            records_read = source_df.count()

            print(
                f"Records read: {records_read}"
            )


            # Add Bronze technical metadata.
            bronze_df = (
                source_df
                .withColumn(
                    "_ingestion_timestamp",
                    F.current_timestamp()
                )
                .withColumn(
                    "_source_file_path",
                    F.col("_metadata.file_path")
                )
                .withColumn(
                    "_source_system",
                    F.lit(SOURCE_SYSTEM)
                )
                .withColumn(
                    "_pipeline_run_id",
                    F.lit(PIPELINE_RUN_ID)
                )
            )


            # Write delivery data to Bronze.
            (
                bronze_df.write
                .format("delta")
                .mode("append")
                .saveAsTable(BRONZE_TABLE)
            )

            records_inserted = bronze_df.count()


            # Mark the delivery as successfully processed.
            escaped_path = delivery_path.replace(
                "'",
                "''"
            )

            spark.sql(f"""
                UPDATE {INGESTION_FILE_REGISTRY_TABLE}
                SET
                    ingestion_status = 'SUCCESS',
                    records_read = {records_read},
                    records_inserted = {records_inserted},
                    ingestion_end_timestamp = current_timestamp(),
                    updated_timestamp = current_timestamp()
                WHERE source_file_path = '{escaped_path}'
                  AND pipeline_run_id = '{PIPELINE_RUN_ID}'
                  AND ingestion_status = 'RUNNING'
            """)

            records_read_total += records_read

            records_inserted_total += records_inserted

            print(
                f"Delivery processed successfully: "
                f"{delivery_name}"
            )


        except Exception as delivery_error:

            error_message = (
                str(delivery_error)
                .replace("'", "''")
            )

            escaped_path = delivery_path.replace(
                "'",
                "''"
            )

            # Mark this specific delivery as FAILED.
            spark.sql(f"""
                UPDATE {INGESTION_FILE_REGISTRY_TABLE}
                SET
                    ingestion_status = 'FAILED',
                    error_message = '{error_message}',
                    ingestion_end_timestamp = current_timestamp(),
                    updated_timestamp = current_timestamp()
                WHERE source_file_path = '{escaped_path}'
                  AND pipeline_run_id = '{PIPELINE_RUN_ID}'
                  AND ingestion_status = 'RUNNING'
            """)

            print(
                f"Delivery failed: {delivery_name}"
            )

            raise


    print(
        f"\nBronze delivery ingestion completed successfully.\n"
        f"Total records read: {records_read_total}\n"
        f"Total records inserted: {records_inserted_total}"
    )


except Exception as pipeline_error:

    error_message = str(pipeline_error)

    print(
        f"Pipeline failed: {error_message}"
    )

    raise

In [0]:
spark.sql(
    f"TRUNCATE TABLE {BRONZE_TABLE}"
)

print("Bronze table cleared for controlled delivery ingestion testing.")

In [0]:
spark.sql(
    f"TRUNCATE TABLE {INGESTION_FILE_REGISTRY_TABLE}"
)

print("Ingestion registry cleared for controlled testing.")

In [0]:
spark.table(BRONZE_TABLE).count()

In [0]:
display(
    spark.sql(f"""
        SELECT
            source_file_name,
            source_file_path,
            ingestion_status,
            records_read,
            records_inserted,
            pipeline_run_id,
            ingestion_start_timestamp,
            ingestion_end_timestamp,
            error_message
        FROM {INGESTION_FILE_REGISTRY_TABLE}
        ORDER BY ingestion_start_timestamp DESC
    """)
)

In [0]:
spark.table(BRONZE_TABLE).count()